In [ ]:
import os
import json
import openai
from PIL import Image
from tqdm import tqdm
import shutil
import cv2
import numpy as np

openai.api_key = os.getenv("OPENAI_API_KEY")

# Paths
image_dir = "/Users/wt/PythonProjects/MultimodalComicAgent/dataset/pororo/Scenes_Dialogues"
qa_json = "/Users/wt/PythonProjects/MultimodalComicAgent/dataset/pororo/qa.json"
output_img_dir = "output_frames"
output_img_json = "output_vqa.json"

os.makedirs(output_img_dir , exist_ok=True)

new_vqa_data = []

# Load dataset
with open(qa_json, "r") as f:
    qa_data = json.load(f)["PororoQA"]

def get_prediction(question, image_paths):
    images = []
    for image_path in image_paths:
        with open(image_path, "rb") as image_file:
            images.append(image_file.read())

    # Call OpenAI API (this is a placeholder, adjust according to actual API usage)
    response = openai.Image.create(
        prompt=question,
        n=1,
        images=images
    )
    return response['data'][0]['text']

correct_count = 0
total_count = len(qa_data)

for entry in tqdm(qa_data):
    video_name = entry["video_name"]
    question = entry["question"]
    correct_idx = entry["correct_idx"]
    answers = [entry[f"answer{i}"] for i in range(5)]
    answer = answers[correct_idx]

    video_folder = video_name.split("_ep")[0]
    gif_folder = os.path.join(image_dir, video_folder, video_name)
    if not os.path.exists(gif_folder):
        print(f"{gif_folder} not exists")
        continue

    frame_files = sorted([f for f in os.listdir(gif_folder) if f.endswith(".gif")], key=lambda x: int(os.path.splitext(x)[0]))

    for frame_file in frame_files:
        gif_path = os.path.join(gif_folder, frame_file)

        try:
            # Open GIF using PIL
            with Image.open(gif_path) as gif:
                total_frames = gif.n_frames
                frame_indices = [0, total_frames // 2, total_frames - 1]  # Select first, middle, and last frames

                for idx in frame_indices:
                    gif.seek(idx)
                    frame = gif.convert("RGB")

                    # Save the extracted frame
                    output_image_path = os.path.join(output_img_dir, f"{video_name}_{frame_file.replace('.gif', f'_{idx}.jpg')}")
                    frame.save(output_image_path)

                    # Append to the new dataset
                    new_vqa_data.append({
                        "image": os.path.basename(output_image_path),
                        "question": question,
                        "answer": answer
                    })
        except Exception as e:
            print(f"Error processing {gif_path}: {e}")
            with open("error_log.txt", "a") as log_file:
                log_file.write(f"Error processing {gif_path}: {e}\n")

# Save new VQA dataset
with open(output_img_json, "w") as f:
    json.dump(new_vqa_data, f, indent=4)

print("Dataset conversion complete!")